# Prototype V1: Semantic Segmentation model
Modelling for green surface detection via NDVI mask and building mask raster calculations. The actual raster calculations are to be run in an additional Jupyter notebook (raster_calculations.ipynb):  
- Building mask: Rasterizing the vector layer of building footprints
- Vegetation mask: Creating a mask based on a NDVI threshold (still need to research on the optimal threshold for this !!!!)  
- Multiplication Building mask * vegetation mask = only vegetation on buildings  
- Addition only vegetation on buildings mask + building mask  

Training data is also created in another Jupyter notebook (i ran it in colab bc my local computer was overwhelmed and the Kernel kept crashing down...).
This notebook is the prototype running only on a subset from the originally chosen 1km x 1km tile.  

So please check beforehand, that the correct training data is created!

----
#### Overall Idea/Concept
Input: Aerial vexcel imagery  
Labels: Combined raster of building and vegetation mask  
Output: a pixel-wise map of green roofs/vegetated roof areas  

---
#### Current possible limitations *(still need to be checked !!!)*
- overhaning trees  
- slight raster alignment errors  
=> how will i deal with this?

#### 0) Colab setup and library loading
Don't forget to first connect to T4 (faster running CPU ?) Important to run those three coding cells for running the entire model in Colab !  
First step with pip install often takes a while to setup, be patient...  
When mounting Google Drive, connect to your Google drive with the 3rd code cell, you might need to log into your Google account and allow to access Colab.

In [ ]:
# Install geospatial libraries
%pip install -q requests geopandas pillow rasterio rioxarray geoai-py leafmap matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [16]:
# loading all required libraries
from pathlib import Path
import geoai
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import geopandas as gpd
import rasterio
from rasterio.windows import Window
import numpy as np

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Change project folder/adapt path
DATA_DIR = Path("/content/drive/MyDrive")

Mounted at /content/drive


---
The following workflow is based on the one proposed in the Gitbook from SDS320 L07 Semantic Segmentation - Binary Segmentation Workflow.
### 1) Loading and Inspecting data
Must first define the training data paths and then inspect their info....

In [2]:
# define path !!!!!!!
# need to run the generate training data notebook first to generate the training data
train_ortho = "../data/training/images/"
train_mask = "../data/training/labels/"

output_path = "../models/"

In [ ]:
# Inspect one ortho raster tile
geoai.print_raster_info(train_ortho + "tile_000001.tif", show_preview=False)

===== RASTER INFORMATION: ../data/training/images/tile_000001.tif =====
Driver: GTiff
Dimensions: 512 x 512 pixels
Number of bands: 4
Data type: uint8
Coordinate Reference System: EPSG:2056
Georeferenced Bounds: BoundingBox(left=2666278.8499999996, bottom=1210711.55, right=2666317.2499999995, top=1210749.95)
Pixel Resolution: 0.07, 0.07
NoData Value: None
----- Band Statistics -----
Band 1:
  Min: 14.00
  Max: 236.00
  Mean: 120.34
  Std Dev: 52.67
Band 2:
  Min: 25.00
  Max: 240.00
  Mean: 126.69
  Std Dev: 56.12
Band 3:
  Min: 18.00
  Max: 241.00
  Mean: 126.47
  Std Dev: 63.13
Band 4:
  Min: 29.00
  Max: 245.00
  Mean: 156.69
  Std Dev: 43.11


In [9]:
# Inspect one mask raster tile
geoai.print_raster_info(train_mask + "tile_000001.tif", show_preview=False)

===== RASTER INFORMATION: ../data/training/labels/tile_000001.tif =====
Driver: GTiff
Dimensions: 512 x 512 pixels
Number of bands: 1
Data type: uint8
Coordinate Reference System: EPSG:2056
Georeferenced Bounds: BoundingBox(left=2666278.8499999996, bottom=1210711.55, right=2666317.2499999995, top=1210749.95)
Pixel Resolution: 0.07, 0.07
NoData Value: None
----- Band Statistics -----
Band 1:
  Min: 0.00
  Max: 2.00
  Mean: 0.02
  Std Dev: 0.14


In [ ]:
# Check Class balance of entire training mask
### Should only include 0 = background; 1 = building without vegetation; 2 = building with vegetation
mask_path = "../data/processed/mask_subset_center01.tif"
with rasterio.open(mask_path) as src:
    mask = src.read(1, masked=True)

valid_pixels = mask.compressed()
unique, counts = np.unique(valid_pixels, return_counts=True)
total = valid_pixels.size

print(f"Unique classes ({len(unique)}): {unique}")
for u, c in zip(unique, counts):
    print(f"Class {u}: {c:,} pixels ({100 * c / total:.2f}%)")

Unique classes (3): [0 1 2]
Class 0: 29,867,433 pixels (67.20%)
Class 1: 13,796,805 pixels (31.04%)
Class 2: 784,651 pixels (1.77%)


Notes after checking data characteristics: Seems in order (but didn't check visually, still need to do that...). Also very small amount of vegetation detected on buildings/roofs, wonder whether the model will actually be able to detect this...

---
### 2) Training U-Net semantic segmentation model
Based on examples from the Gibook Chapter L07

In [ ]:
# Defining a consistent legend for multiclass segmentation
### for now only [0 1 2], maybe later also categorize different subtypes of green roofing like in STDL paper
### For that, adjust this cell then

### Evtl unnötig weil eh nur 3 Klassen....
CLASS_NAMES = {
    0: "Background",
    1: "Building",
    2: "Building with vegetated roof"
}

CLASS_COLORS = {
    0: "#000000",
    1: "#BDBDBB",
    2: "#048209"
}

class_ids = sorted(CLASS_NAMES)
assert set(CLASS_COLORS) == set(class_ids), "Every class must have a color."

LC_CMAP = ListedColormap([CLASS_COLORS[i] for i in class_ids])
class_ids_array = np.asarray(class_ids)
class_boundaries = np.r_[
    class_ids_array[0] - 0.5,
    (class_ids_array[:-1] + class_ids_array[1:]) / 2,
    class_ids_array[-1] + 0.5,
]
LC_NORM = BoundaryNorm(class_boundaries, LC_CMAP.N)
LEGEND = [
    Patch(facecolor=CLASS_COLORS[i], edgecolor="gray", label=f"{i}: {CLASS_NAMES[i]}")
    for i in class_ids
]

# Check that the class mapping matches the values in the raster
with rasterio.open(mask_path) as src:
    raster_classes = np.unique(src.read(1, masked=True).compressed())

assert np.array_equal(raster_classes, class_ids_array), (
    f"Raster classes {raster_classes.tolist()} do not match palette classes {class_ids}."
)
print(f"Class mapping check passed: {raster_classes.tolist()}")

Class mapping check passed: [0, 1, 2]


In [ ]:
# Train U-Net model
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

In [ ]:
# Evaluate Model
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
)

In [ ]:
# Run inference on test raster
masks_path = "marly_buildings_prediction.tif"
probability_path = "marly_test_probability_map.tif"
model_path = f"{out_folder}/unet_models/best_model.pth"

geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    probability_path=probability_path,
)

In [ ]:
# Inspect masks and ucertainty
geoai.view_raster(
    masks_path,
    nodata=0,
    colormap="binary",
    basemap=test_raster_path,
)

In [ ]:
geoai.view_raster(
    probability_path,
    indexes=[2],
    basemap=test_raster_path,
)

In [ ]:
# Vectorize and filter predictions
output_vector_path = "marly_buildings_prediction.geojson"

gdf = geoai.orthogonalize(
    masks_path,
    output_vector_path,
    epsilon=2,
)

gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")
gdf_filtered = gdf_props[(gdf_props["area_m2"] > 10)]

geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_path,
)